# REINFORCE in a Real Environment

Collect CartPole episodes, compute returns, and perform policy updates with honest evaluation.

## Before You Start

Course 12. Entry course: 10. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** rl-02_trajectory.

## What You Will Be Able to Do

- Compute reward-to-go
- Separate termination and time-limit truncation
- Run and evaluate on-policy CartPole learning

## A complete loop

Reset the environment with a recorded seed, observe s, sample an action from the current policy, step the environment, and store the chosen-action log probability and reward. Stop on either terminated or truncated. CartPole-v1 terminates when the pole or cart leaves its allowed range and truncates at its time limit. A time limit is not inherently a physical terminal state; value-bootstrapped algorithms must treat that distinction carefully.

Compute G_t=r_t+gamma*G_(t+1) backward from the last reward. For the objective E[sum_t gamma^t r_t], the exact episodic score-function form uses gamma^t*G_t at time t. Our lab includes this outer factor. Many practical implementations use different weighting conventions; name the objective rather than mixing formulas. The loss to minimize is minus the mean across episodes of the sum of log probabilities times these discounted advantages.

## Learning and evaluating are different modes

A baseline may use a running return estimate from previous episodes or a learned state-value function; detach it in the policy loss. Normalizing advantages can stabilize practical training but changes the finite-sample estimator, so disclose it. Reward-to-go reduces irrelevant past-reward variance. Entropy bonuses can encourage exploration but can also reduce short-term reward.

The notebook uses Gymnasium's established CartPole physics, a small Torch policy, real sampled rollouts, and real backward/optimizer steps. It reports pre/post evaluation on fixed separate seeds, episode lengths, and every training seed. The default short run proves the pipeline executes; it does not promise to solve CartPole. No graphical display or GPU is required. Increase the episode budget as an experiment, not a hidden requirement for the browser challenge.

## Worked Example: Reward-to-go

Rewards=[1,0,2], gamma=.5.

1. **Last step:** G2=2.
2. **Middle:** G1=0+.5*2=1.
3. **First:** G0=1+.5*1=1.5. The objective weights for the score terms are [1.5,.5,.5] after multiplying by gamma^t.

Relative reward-to-go and the full discounted-objective weighting are not identical arrays.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Episodic discounted REINFORCE objective."""
import numpy as np

def discounted_returns(rewards, gamma):
    """Return G_t=sum_{k>=t} gamma^(k-t)*r_k for finite 1D rewards.

Require finite 0<=gamma<=1, else ValueError. Empty sequence returns empty."""
    raise NotImplementedError('Implement this function, then run the checks.')

def reinforce_loss(log_probabilities, rewards, mask, gamma=1.0, baseline=0.0):
    """Mean episode loss for discounted initial-state return.

Matching [B,T] log probabilities/rewards and Boolean mask. Masks are
nonempty true prefixes followed by padding. Assume finite valid arrays.
L=-mean_b sum_t gamma^t*logp_bt*(G_bt-baseline). Baseline is a fixed scalar,
not differentiated. Ignore all padding; do not average across token count."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

Rewards=[1,1,1], gamma=1.

- Return at time 0

### Worked Answers

- 3: All three undiscounted rewards count.

## Experiment

Run `experiments/rl_cartpole.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Compare evaluation before and after actual updates on separate seeds. Report reward distributions, not just a maximum.
- Map CartPole state/action/episode/reward to text prefix/next token/completion/verifier score. The mathematics transfers; the environment and failure modes differ.

## Transfer Problems

1. Compare a running baseline with zero baseline at equal sampled transitions.
2. Replace dense step rewards with a sparse success reward and measure variance.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[Gymnasium: CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.


## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3" "gymnasium>=1,<2"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Episodic discounted REINFORCE objective."""
import numpy as np

def discounted_returns(rewards, gamma):
    """Return G_t=sum_{k>=t} gamma^(k-t)*r_k for finite 1D rewards.

Require finite 0<=gamma<=1, else ValueError. Empty sequence returns empty."""
    raise NotImplementedError('Implement this function, then run the checks.')

def reinforce_loss(log_probabilities, rewards, mask, gamma=1.0, baseline=0.0):
    """Mean episode loss for discounted initial-state return.

Matching [B,T] log probabilities/rewards and Boolean mask. Masks are
nonempty true prefixes followed by padding. Assume finite valid arrays.
L=-mean_b sum_t gamma^t*logp_bt*(G_bt-baseline). Baseline is a fixed scalar,
not differentiated. Ignore all padding; do not average across token count."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
assert np.allclose(discounted_returns([1,0,2], .5), [1.5,1,2])
assert np.allclose(discounted_returns([1,2], 0), [1,2])
assert len(discounted_returns([], 1)) == 0
lp = np.log(np.array([[.5,.25,.75],[.8,.2,.9]]))
r = np.array([[1.,0.,2.],[3.,999.,999.]])
m = np.array([[1,1,1],[1,0,0]], bool)
expected = -(np.dot(lp[0], np.array([1.5,.5,.5])) + lp[1,0]*3)/2
assert np.isclose(reinforce_loss(lp,r,m,.5), expected), 'Episode weighting and discount factors'
changed = lp.copy(); changed[~m] = -999
assert np.isclose(reinforce_loss(changed,r,m,.5), expected)
print('REINFORCE checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Episodic discounted REINFORCE objective."""
import numpy as np


def discounted_returns(rewards, gamma):
    """Return G_t=sum_{k>=t} gamma^(k-t)*r_k for finite 1D rewards.

    Require finite 0<=gamma<=1, else ValueError. Empty sequence returns empty.
    """
    if not np.isfinite(gamma) or not 0 <= gamma <= 1:
        raise ValueError('invalid discount')
    result = np.empty(len(rewards), float); running = 0.
    for t in range(len(rewards)-1, -1, -1):
        running = rewards[t] + gamma*running
        result[t] = running
    return result


def reinforce_loss(log_probabilities, rewards, mask, gamma=1., baseline=0.):
    """Mean episode loss for discounted initial-state return.

    Matching [B,T] log probabilities/rewards and Boolean mask. Masks are
    nonempty true prefixes followed by padding. Assume finite valid arrays.
    L=-mean_b sum_t gamma^t*logp_bt*(G_bt-baseline). Baseline is a fixed scalar,
    not differentiated. Ignore all padding; do not average across token count.
    """
    losses = []
    for logp, r, valid in zip(log_probabilities, rewards, mask):
        count = int(valid.sum())
        returns = discounted_returns(r[:count], gamma)
        losses.append(-np.sum(gamma**np.arange(count)*logp[:count]*(returns-baseline)))
    return float(np.mean(losses))


In [ ]:
import numpy as np
assert np.allclose(discounted_returns([1,0,2], .5), [1.5,1,2])
assert np.allclose(discounted_returns([1,2], 0), [1,2])
assert len(discounted_returns([], 1)) == 0
lp = np.log(np.array([[.5,.25,.75],[.8,.2,.9]]))
r = np.array([[1.,0.,2.],[3.,999.,999.]])
m = np.array([[1,1,1],[1,0,0]], bool)
expected = -(np.dot(lp[0], np.array([1.5,.5,.5])) + lp[1,0]*3)/2
assert np.isclose(reinforce_loss(lp,r,m,.5), expected), 'Episode weighting and discount factors'
changed = lp.copy(); changed[~m] = -999
assert np.isclose(reinforce_loss(changed,r,m,.5), expected)
print('REINFORCE checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Real Gymnasium CartPole rollouts and REINFORCE updates on CPU.

uv run --with numpy --with torch --with gymnasium python experiments/rl_cartpole.py
No rendering dependency. Default 60 episodes is an execution study, not a solve.
"""
import json
import os
import gymnasium as gym
import numpy as np
import torch
from torch import nn


def cartpole_experiment(episodes=60):
    torch.set_num_threads(1)
    reports = []
    for seed in [4, 13]:
        torch.manual_seed(seed)
        policy = nn.Sequential(nn.Linear(4, 16), nn.Tanh(), nn.Linear(16, 2))
        optimizer = torch.optim.Adam(policy.parameters(), lr=.008)
        env = gym.make('CartPole-v1')
        def evaluation():
            lengths = []
            for eval_seed in [1001,1002,1003,1004,1005]:
                state, _ = env.reset(seed=eval_seed); count = 0
                while True:
                    with torch.no_grad():
                        action = int(policy(torch.tensor(state)).argmax())
                    state, _, terminated, truncated, _ = env.step(action); count += 1
                    if terminated or truncated:
                        break
                lengths.append(count)
            return lengths
        before = evaluation(); curve = []; baseline = 0.; sampled_transitions = 0
        initial_parameters = [p.detach().clone() for p in policy.parameters()]
        for episode in range(episodes):
            state, _ = env.reset(seed=seed*1000+episode)
            logps, rewards = [], []
            while True:
                distribution = torch.distributions.Categorical(logits=policy(torch.tensor(state)))
                action = distribution.sample()
                logps.append(distribution.log_prob(action))
                state, reward, terminated, truncated, _ = env.step(int(action)); rewards.append(reward)
                if terminated or truncated:
                    break
            gamma = .99; running = 0.; returns = []
            for reward in reversed(rewards):
                running = reward + gamma*running; returns.append(running)
            returns = torch.tensor(list(reversed(returns)))
            weights = gamma**torch.arange(len(rewards))
            loss = -(torch.stack(logps)*weights*(returns-baseline)).sum()
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 10.); optimizer.step()
            baseline = .9*baseline+.1*float(returns.mean())
            sampled_transitions += len(rewards)
            curve.append(len(rewards))
        after = evaluation(); env.close()
        assert any(not torch.equal(a,b) for a,b in zip(initial_parameters,policy.parameters()))
        reports.append(dict(seed=seed, episodes=episodes, sampled_transitions=sampled_transitions,
            before_eval_lengths=before, after_eval_lengths=after, training_lengths=curve,
            before_mean=float(np.mean(before)), after_mean=float(np.mean(after)),
            limitation='Greedy evaluation on five fixed seeds; no claim of solving the environment. No truncation bootstrap in this finite-episode Monte Carlo objective.'))
    return reports


if __name__ == '__main__':
    print(json.dumps(cartpole_experiment(int(os.environ.get('CARTPOLE_EPISODES', '60'))), indent=2))
